# GPU-Streaming-Test: Drive → Colab → GitHub → Drive
**GPU-Upscaling:** Die NVIDIA T4 übernimmt mit `scale_cuda` das Skalieren auf 3840×2160 sowie die H.264-Kodierung mit `h264_nvenc`. FFmpeg prüft den CUDA-Filter vor dem Start. Die fertigen 10-Sekunden-Abschnitte werden zu GitHub-Releases hochgeladen; GitHub Actions lädt das Ergebnis zu Drive. Colab-Secret `GH_TOKEN` erforderlich. **Hinweis:** Diese Version gilt erst nach dem erneuten Öffnen/Starten des Notebooks; ein bereits laufender Job verwendet weiterhin den alten Code.


In [ ]:
# Stream-Input von Drive (ohne 11-GB-Download) -> NVENC -> GitHub Release Assets
# Erforderlich: Colab Secrets -> GH_TOKEN (GitHub Fine-grained PAT mit Contents: read/write)
!pip -q install google-api-python-client requests
from google.colab import auth, userdata
auth.authenticate_user()
import google.auth, google.auth.transport.requests, requests, subprocess, time
from pathlib import Path
from urllib.parse import quote
creds,_=google.auth.default(scopes=['https://www.googleapis.com/auth/drive.readonly'])
creds.refresh(google.auth.transport.requests.Request())
token=creds.token
gh=userdata.get('GH_TOKEN')
if not gh: raise RuntimeError('Colab Secret GH_TOKEN fehlt. Bitte nicht ins Notebook schreiben.')
repo='nikl223-ctrl/LibreMediaConverter'
headers={'Authorization':'Bearer '+gh,'Accept':'application/vnd.github+json','X-GitHub-Api-Version':'2022-11-28'}
tag='colab-gpu-'+time.strftime('%Y%m%d-%H%M%S')
api='https://api.github.com/repos/'+repo
work=Path('/content/gpu_parts'); work.mkdir(exist_ok=True)
def request(method,url,**kw):
    r=requests.request(method,url,headers=headers,timeout=180,**kw)
    if not r.ok:
        try: detail=r.json()
        except ValueError: detail={'message':r.text[:500]}
        print('GitHub-Fehler:',r.status_code,detail.get('message','Unbekannt'))
        print('Details:',str(detail.get('errors',''))[:600])
        if r.status_code==403:
            print('Prüfe Fine-grained PAT: Resource owner nikl223-ctrl, Repository LibreMediaConverter, Contents Read and write. Falls GitHub SSO/Approval verlangt, autorisieren.')
        r.raise_for_status()
    return r
print('Prüfe GitHub-Berechtigungen vor dem GPU-Start...')
who=request('GET','https://api.github.com/user').json()
print('Token gehört zu:',who.get('login','unbekannt'))
info=request('GET',api).json()
print('Repository:',info.get('full_name'), 'Push erlaubt:',info.get('permissions',{}).get('push','nicht angegeben'))
print('GPU und NVENC prüfen...')
subprocess.run(['nvidia-smi'],check=True)
enc=subprocess.run(['ffmpeg','-hide_banner','-encoders'],capture_output=True,text=True,check=True).stdout
if 'h264_nvenc' not in enc: raise RuntimeError('h264_nvenc fehlt.')
filters=subprocess.run(['ffmpeg','-hide_banner','-filters'],capture_output=True,text=True,check=True).stdout
if 'scale_cuda' not in filters:
    raise RuntimeError('Der FFmpeg-Build enthält scale_cuda nicht. GPU-Upscaling benötigt einen FFmpeg-Build mit CUDA-Filtern.')
print('NVIDIA NVENC und CUDA-Upscaling (scale_cuda) verfügbar.')
# GitHub Release zuerst als Draft erstellen, erst nach 6 Uploads veröffentlichen.
release=request('POST',api+'/releases',json={'tag_name':tag,'target_commitish':'main','name':tag,'draft':True,'prerelease':True}).json()
release_id=release['id']
upload_url=release['upload_url'].split('{')[0]
drive_url='https://www.googleapis.com/drive/v3/files/1_fD9iNP8uDkV81L1ne-03sTEa6ERj1Ol?alt=media'
for i in range(6):
    output=work/f'part_{i:02d}.mp4'
    # Drive API unterstützt HTTP Range; FFmpeg liest per HTTP, ohne die ganze Datei zu speichern.
    command=['ffmpeg','-hide_banner','-y','-loglevel','warning',
      '-headers','Authorization: Bearer '+token+'\r\n',
      '-ss',str(i*10),'-i',drive_url,'-t','10',
      '-map','0:v:0','-map','0:a:0?',
      '-hwaccel','cuda','-hwaccel_output_format','cuda',
      '-vf','scale_cuda=w=3840:h=2160:format=nv12',
      '-c:v','h264_nvenc','-b:v','20M','-maxrate','20M','-bufsize','40M',
      '-c:a','aac','-b:a','384k','-ac','6','-movflags','+faststart',str(output)]
    print('Konvertiere Abschnitt',i+1,'/6')
    subprocess.run(command,check=True)
    print('Übertrage Abschnitt an GitHub:',output.name,round(output.stat().st_size/1048576,1),'MiB')
    # Upload per Release Asset: Github speichert Segmente nur vorübergehend.
    with output.open('rb') as stream:
        h=dict(headers);h['Content-Type']='video/mp4'
        res=requests.post(upload_url,params={'name':output.name},headers=h,data=stream,timeout=900)
        res.raise_for_status()
    output.unlink()
print('Alle Abschnitte übertragen. Starte GitHub Actions durch Veröffentlichung...')
request('PATCH',api+'/releases/'+str(release_id),json={'draft':False})
print('GitHub-Workflow:',f'https://github.com/{repo}/actions')
print('Release:',f'https://github.com/{repo}/releases/tag/{tag}')
print('Ergebnis wird von GitHub Actions nach Google Drive hochgeladen, falls die Drive-Secrets eingerichtet sind.')
